<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:22px 28px; border-radius:8px">
  <div style="font-size:0.9em; letter-spacing:2px; opacity:0.85">NOTEBOOK 04 · NLP EXTRACTION · 🧠 THE GENAI CORE · COMPLETED</div>
  <div style="font-size:2.0em; font-weight:700; margin-top:4px">🧠 Reading the notes with <code>ai_query</code></div>
  <div style="font-size:1.1em; margin-top:8px; max-width:880px; opacity:0.95">
    Recover HER2 / ER / PR status from free-text pathology reports with a Foundation Model,
    and pull the ~60 "notes-only" patients back into the cohort that structured SQL never sees.
  </div>
</div>

## Why SQL can't do this, but a Foundation Model can

Notebook 03 surfaced the gap: ~**60 patients** (person 181 to 240) have their biomarker status
written *only* in a free-text pathology note. They have **zero** rows in `measurement`, so the
silver pivot (and every SQL cohort query) returns nothing for them.

Pathologists write HER2/ER/PR a dozen different ways:

| What the note says | Means |
|---|---|
| `HER2: Positive (3+ by IHC)` | HER2 **Positive** |
| `Her-2/neu overexpression confirmed: 3+` | HER2 **Positive** |
| `HER2/neu amplification: NOT detected (FISH ratio 1.4)` | HER2 **Negative** |
| `HER2 (c-erbB-2): 2+ (equivocal by IHC; FISH reflex recommended)` | HER2 **Unknown / equivocal** |

No `LIKE '%positive%'` survives that ("NOT detected" even contains "detected"). But a Foundation
Model reads the line like a clinician. We call it **from SQL** with **`ai_query`**: no model
hosting, no Python serving code, full Unity Catalog lineage.

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## 🔑 The pattern (read this before the calls below)

`ai_query(endpoint, prompt, responseFormat => ...)` calls a model on each row. Two pieces work
together so the output is parseable:

1. **`responseFormat`** turns on *constrained decoding* so the model emits **clean JSON**, not
   `` ```json … ``` `` markdown fences. Without it, `from_json` silently returns **NULL** for every
   fenced row and your whole column comes back empty. (This was a real bug in the original build.)
2. **`from_json`** turns that JSON string into typed columns you can select.

<div style="background:#FFF8E1; border-left:6px solid #F2A900; padding:12px 16px; border-radius:4px">
<b>The one gotcha:</b> the <code>responseFormat</code> DDL form allows only <b>one</b> top-level
field, so it is wrapped as <code>STRUCT&lt;result:STRUCT&lt;...&gt;&gt;</code>. The model still emits the
flat keys, which the <code>from_json</code> schema (<code>STRUCT&lt;her2_status:STRING, er_status:STRING,
pr_status:STRING&gt;</code>) matches.
</div>

```
responseFormat: 'STRUCT<result:STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>>'
from_json schema: 'STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>'
endpoint (the _config var):  LLM_FAST  ==  'databricks-claude-haiku-4-5'
```

## 1️⃣ First, just ask the model about ONE note (prove the call works) (COMPLETED)

In [0]:
%sql
SELECT person_id, extracted.her2_status, extracted.er_status, extracted.pr_status, note_text
FROM (
  SELECT person_id, note_text,
    from_json(
      ai_query(
        'databricks-claude-haiku-4-5',
        'Extract the HER2, ER (estrogen receptor), and PR (progesterone receptor) status from this '
        || 'breast cancer pathology report. Respond with exactly one of Positive, Negative, or Unknown '
        || 'for each. Use Unknown if equivocal or not stated. Report: ' || note_text,
        responseFormat => 'STRUCT<result:STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>>'
      ),
      'STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>'
    ) AS extracted
  FROM note
  WHERE note_source_value = 'PATHOLOGY_REPORT'
  LIMIT 1
);

person_id,her2_status,er_status,pr_status,note_text
1,Positive,Positive,Negative,"PATHOLOGY REPORT — CORE NEEDLE BIOPSY PATH-524019 | 2023-01-06 SPECIMEN: Left breast lumpectomy specimen INDICATION: Palpable Left breast mass. Rule out malignancy. HISTOLOGIC FINDINGS: Diagnosis: Invasive ductal carcinoma, grade 2 Nuclear Grade: 2/3 Estimated Tumor Size: 1.6 cm Lymphovascular Invasion: Not identified RECEPTOR / BIOMARKER STATUS: Estrogen receptor: reactive, 95% of nuclei, Allred score 8 Progesterone receptor (PR): Negative (< 1%) HER2: Positive (3+ by IHC) Ki-67: 61% MOLECULAR SUBTYPE: HR+/HER2+ (luminal B-like, HER2-positive) COMMENT: Clinical correlation recommended. These findings represent a biopsy diagnosis; final staging will require complete excision and axillary lymph node evaluation. Pathologist: Jennifer Hopkins, M.D. | Meridian Cancer Center"


## 2️⃣ Run it across every pathology note → `silver_nlp_biomarkers` (COMPLETED)

Same call, no `LIMIT`. Materialize the result as **`silver_nlp_biomarkers`**: one row per patient
with a pathology report, the three extracted statuses, and a literal `biomarker_source = 'nlp'` so
it sits cleanly alongside the structured `silver_biomarker_profile` from nb 02.

In [0]:
%sql
CREATE OR REPLACE TABLE silver_nlp_biomarkers
COMMENT 'HER2/ER/PR read from free-text pathology notes by a Foundation Model (ai_query). biomarker_source = nlp.'
AS
SELECT person_id,
       extracted.her2_status AS her2_status,
       extracted.er_status   AS er_status,
       extracted.pr_status   AS pr_status,
       'nlp'                 AS biomarker_source
FROM (
  SELECT person_id,
    from_json(
      ai_query(
        'databricks-claude-haiku-4-5',
        'Extract the HER2, ER (estrogen receptor), and PR (progesterone receptor) status from this '
        || 'breast cancer pathology report. Respond with exactly one of Positive, Negative, or Unknown '
        || 'for each. Use Unknown if equivocal or not stated. Report: ' || note_text,
        responseFormat => 'STRUCT<result:STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>>'
      ),
      'STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>'
    ) AS extracted
  FROM note
  WHERE note_source_value = 'PATHOLOGY_REPORT'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT person_id, her2_status, er_status, pr_status, biomarker_source
FROM silver_nlp_biomarkers ORDER BY person_id LIMIT 10;

person_id,her2_status,er_status,pr_status,biomarker_source
1,Positive,Positive,Negative,nlp
2,Positive,Negative,Positive,nlp
3,Positive,Negative,Negative,nlp
4,Positive,Positive,Negative,nlp
5,Positive,Negative,Positive,nlp
6,Positive,Negative,Positive,nlp
7,Positive,Positive,Negative,nlp
8,Positive,Negative,Negative,nlp
9,Positive,Positive,Positive,nlp
10,Positive,Negative,Positive,nlp


## 3️⃣ Did we recover the invisible patients? 🎯 (PRE-BUILT proof)

The whole point. A "notes-only" patient has a `PATHOLOGY_REPORT` note but **no** structured
`measurement` rows. Count how many now have an NLP biomarker call.

In [0]:
%sql
WITH has_struct AS (
  SELECT DISTINCT person_id FROM measurement
  WHERE measurement_source_value IN ('HER2/neu','Estrogen receptor','Progesterone receptor')
),
notes_only AS (
  SELECT DISTINCT person_id FROM note
  WHERE note_source_value = 'PATHOLOGY_REPORT'
    AND person_id NOT IN (SELECT person_id FROM has_struct)
)
SELECT
  COUNT(*)                                                          AS notes_only_patients,
  SUM(CASE WHEN nlp.person_id IS NOT NULL THEN 1 ELSE 0 END)        AS recovered_by_nlp,
  SUM(CASE WHEN nlp.her2_status IN ('Positive','Negative') THEN 1 ELSE 0 END) AS with_definite_her2
FROM notes_only n
LEFT JOIN silver_nlp_biomarkers nlp ON n.person_id = nlp.person_id;

notes_only_patients,recovered_by_nlp,with_definite_her2
60,60,60


Expected: **60 notes-only patients, ~60 recovered, ~100% with a definite HER2 call.**

## 4️⃣ Quick accuracy gut-check (PRE-BUILT, the rigorous eval is nb 07)

For the "both" patients we can hold the model's reading up against the structured ground truth.
This is a sanity check; the full scored evaluation lives in **notebook 07**.

<div style="background:#FFF8E1; border-left:6px solid #F2A900; padding:10px 14px; border-radius:4px">
Agreement here is <b>high but not 100%</b>, and that is by design. The foundation plants a
hard-case band (person 61-90) whose notes are written equivocally (HER2 IHC 2+ with a reflex FISH
ratio, ER-low-positive). A quick prompt reads some of those as "Unknown", so they show up as
disagreements. That is exactly the contrast notebook 07 measures.
</div>

In [0]:
%sql
WITH struct_her2 AS (
  SELECT person_id, MAX(value_source_value) AS her2_structured
  FROM measurement WHERE measurement_source_value = 'HER2/neu' GROUP BY person_id
)
SELECT
  COUNT(*)                                                                  AS both_patients,
  SUM(CASE WHEN s.her2_structured = nlp.her2_status THEN 1 ELSE 0 END)       AS agree,
  ROUND(100.0 * AVG(CASE WHEN s.her2_structured = nlp.her2_status THEN 1 ELSE 0 END), 1) AS agree_pct
FROM struct_her2 s
JOIN silver_nlp_biomarkers nlp ON s.person_id = nlp.person_id;

both_patients,agree,agree_pct
180,164,91.1


<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px">
<b>What we just did:</b> turned unstructured pathology text into typed HER2/ER/PR columns with a
single <code>ai_query</code>: no model serving, no Python, full lineage in Unity Catalog. That
recovered the notes-only patients SQL could never see. <b>This is the NLP value moment.</b>
</div>

## ▶️ Next step
### → Open **[05_clinicalbert_mlflow_uc]($./05_clinicalbert_mlflow_uc)** to register a domain ClinicalBERT model to Unity Catalog and serve it on an endpoint.